In [20]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import remove_stopwords
from gensim.utils import simple_preprocess
import re
import pandas as pd
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [21]:
df=pd.read_csv('IMDB Dataset.csv')
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})

In [22]:
def clean_review(text):
    text=re.sub(r'<.*>',' ',text)
    text=remove_stopwords(text.lower())
    return simple_preprocess(text)
df['tokenized_reviews']=df['review'].apply(clean_review)

In [23]:
x_train,x_test,y_train,y_test=train_test_split(df['tokenized_reviews'].tolist(),
                                              df['sentiment'].values,
                                              test_size=0.2,
                                              random_state=42)

In [24]:
w2v_model=Word2Vec(sentences=x_train,vector_size=100,window=5,min_count=2,sg=0)

In [25]:
def get_document_embedding(doc,model):
    valid_words=[word for word in doc if word in model.wv]

    if len(valid_words)==0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_words],axis=0)


In [26]:
x_train_emb=np.array([
    get_document_embedding(doc,w2v_model)
    for doc in x_train
])
x_test_emb=np.array([
    get_document_embedding(doc,w2v_model)
    for doc in x_test
])

In [27]:
model=Sequential([
    Input(shape=(x_train_emb.shape[1],)),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [28]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [30]:
history=model.fit(x_train_emb,y_train,epochs=10,batch_size=128,validation_split=0.1)

Epoch 1/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7789 - loss: 0.4695 - val_accuracy: 0.8117 - val_loss: 0.4188
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8054 - loss: 0.4274 - val_accuracy: 0.8110 - val_loss: 0.4105
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8096 - loss: 0.4194 - val_accuracy: 0.8002 - val_loss: 0.4245
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8109 - loss: 0.4147 - val_accuracy: 0.8117 - val_loss: 0.4077
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8124 - loss: 0.4091 - val_accuracy: 0.8115 - val_loss: 0.4033
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8132 - loss: 0.4063 - val_accuracy: 0.8160 - val_loss: 0.4022
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8173 - loss: 0.4023 - val_accuracy: 0.8158 - val_loss: 0.3993
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8161 - loss: 0.3998 - val_accuracy: 0.

In [31]:
test_loss,test_accuracy=model.evaluate(x_test_emb,y_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8029 - loss: 0.4197
Test Loss: 0.4197
Test Accuracy: 80.29%
